# Pitcher Future Projections - Detailed Workflow

Complete workflow for generating multi-year pitcher WAR projections.

**Purpose:** Pitcher-specific projection pipeline with detailed control and analysis.

**Features:**
- Role-specific aging curves (SP vs RP)
- Pitcher model features (K%, BB%, ERA, FIP, GB%, etc.)
- Elite pitcher protection
- Tommy John surgery recovery adjustments

In [ ]:
# Cell 1: Imports and Setup

import sys
from pathlib import Path
import pandas as pd
import numpy as np

# Add project root to path
project_root = Path('.').absolute().parent.parent
sys.path.insert(0, str(project_root))

# Import pitcher-specific modules
from new_pipeline.models.future_season import (
    FutureProjectionPipeline,
    load_historical_player_data,
    build_longitudinal_sequences
)
from new_pipeline.common.constants import PITCHER_MODEL_FEATURES

print("Pitcher Future Projections - sWARm")
print("=" * 70)
print("Multi-year WAR projections for pitchers")
print()

# Configuration
BASE_YEAR = 2024
YEARS_AHEAD = 3

print(f"Configuration:")
print(f"  Player type: Pitchers")
print(f"  Base year: {BASE_YEAR}")
print(f"  Projection years: {BASE_YEAR + 1} - {BASE_YEAR + YEARS_AHEAD}")
print(f"  Model features: {len(PITCHER_MODEL_FEATURES)}")
print(f"    {', '.join(PITCHER_MODEL_FEATURES)}")

## Step 1: Data Loading and Preparation

In [ ]:
# Cell 2: Load Historical Pitcher Data

print("\nLoading historical pitcher data...")

historical_data = load_historical_player_data(
    player_type='pitcher',
    years=list(range(BASE_YEAR - 8, BASE_YEAR + 1))
)

print(f"  Loaded {len(historical_data)} pitcher-season records")
print(f"  Years: {historical_data['Year'].min()} - {historical_data['Year'].max()}")
print(f"  Unique pitchers: {historical_data['playerid'].nunique()}")

# Show role distribution (SP vs RP)
print("\nRole distribution:")
if 'Role' in historical_data.columns:
    role_counts = historical_data['Role'].value_counts()
    for role, count in role_counts.items():
        print(f"  {role}: {count}")
elif 'Position' in historical_data.columns:
    pos_counts = historical_data['Position'].value_counts()
    for pos, count in pos_counts.items():
        print(f"  {pos}: {count}")

In [ ]:
# Cell 3: Build Longitudinal Sequences

print("\nBuilding longitudinal sequences for pitchers...")

sequences_df = build_longitudinal_sequences(
    historical_data,
    player_type='pitcher'
)

print(f"  Created {len(sequences_df)} sequences")
print(f"  Sequence years: {sequences_df['year_n'].min()} - {sequences_df['year_n'].max()}")
print(f"  Features per sequence: {len(sequences_df.columns)}")

# Show feature summary
print("\nKey features included:")
feature_cols = [col for col in sequences_df.columns if col.endswith('_n')]
print(f"  Total: {len(feature_cols)} features")
print(f"  Sample: {', '.join(feature_cols[:10])}...")

## Step 2: Initialize and Run Projection Pipeline

In [ ]:
# Cell 4: Initialize Pitcher Pipeline

print("\nInitializing pitcher projection pipeline...")

pitcher_pipeline = FutureProjectionPipeline(
    player_type='pitcher',
    base_year=BASE_YEAR,
    years_ahead=YEARS_AHEAD
)

print("\nPipeline components initialized:")
print("  [*] Longitudinal Model - RandomForest with 14 pitcher features")
print("  [*] Survival Model - Cox PH for retirement probability")
print("  [*] Age Curves - Role-specific aging (SP, RP, Swing)")
print("  [*] Elite Adjustments - Protect 6+ WAR pitchers from over-regression")
print("  [*] Injury Recovery - Tommy John and elbow surgery adjustments")

In [ ]:
# Cell 5: Run Full Pipeline

print("\nRunning complete pitcher projection pipeline...")
print("This includes: data loading, model training, projection generation, and adjustments.")
print()

pitcher_projections = pitcher_pipeline.run_full_pipeline(
    injury_records=None,  # Optional: provide Tommy John surgery data if available
    save_output=True
)

print(f"\nPitcher projections complete!")
print(f"  Total pitchers projected: {len(pitcher_projections)}")

In [ ]:
# Cell 7.5: Ensemble Model Breakdown

print("\n" + "=" * 70)
print("ENSEMBLE MODEL BREAKDOWN")
print("=" * 70)

# Access the ensemble model from the pipeline
ensemble_model = pitcher_pipeline.longitudinal_model

# Display ensemble configuration
print("\nEnsemble Configuration:")
print("-" * 70)
print("Adaptive Weighting by Player History Length:")
print("  Veterans (5+ years):     XGB=0.35, RNN=0.35, ExtraTrees=0.30")
print("  Mid-career (3-4 years):  XGB=0.45, RNN=0.25, ExtraTrees=0.30")
print("  Short history (<4 years): Fallback model (ExtraTrees)")

# Count players by history length from historical data
print("\nPlayer Distribution by History Length:")
print("-" * 70)

player_history_lengths = historical_data.groupby('playerid')['Year'].count()
veterans = (player_history_lengths >= 5).sum()
mid_career = ((player_history_lengths >= 3) & (player_history_lengths < 5)).sum()
short_history = (player_history_lengths < 3).sum()

total_players = len(player_history_lengths)
print(f"  Veterans (5+ seasons):    {veterans:4d} ({100*veterans/total_players:5.1f}%)")
print(f"  Mid-career (3-4 seasons): {mid_career:4d} ({100*mid_career/total_players:5.1f}%)")
print(f"  Short history (<3):       {short_history:4d} ({100*short_history/total_players:5.1f}%)")
print(f"  Total players:            {total_players:4d}")

print("\nModel Architecture:")
print("-" * 70)
print("  [1] XGBoost (Darts XGBModel)")
print("      - Gradient boosting with automatic lag creation")
print("      - Uses last 3 years of WAR + features")
print("      - Best for players with stable performance")
print()
print("  [2] RNN (Darts GRU)")
print("      - Learns WAR trajectory patterns")
print("      - Requires 3+ consecutive seasons")
print("      - Best for capturing performance trends")
print()
print("  [3] ExtraTrees (Darts SKLearnModel)")
print("      - Ensemble of decision trees")
print("      - Uses last 3 years of features")
print("      - Robust to outliers")
print()
print("  [4] ExtraTrees Fallback")
print("      - Standard RandomForest")
print("      - For players with <4 seasons or non-consecutive careers")
print("      - Trained on all players (cross-player learning)")

print("\nPitcher-Specific Considerations:")
print("-" * 70)
print("  - Tommy John recovery captured in injury features")
print("  - Role transitions (SP <-> RP) handled via feature history")
print("  - Workload management reflected in IP trends")
print("  - Fallback critical for rookies and injury-prone pitchers")

print("\nEnsemble Benefits:")
print("-" * 70)
print("  - Combines strengths of multiple algorithms")
print("  - Adaptive weighting based on player history")
print("  - Robust to career gaps and injuries")
print("  - No players filtered out (fallback handles all cases)")
print()
print("=" * 70)

## Step 3: Analyze Projections

In [ ]:
# Cell 6: Top Projected Pitchers by Year

print("\nTop 25 Projected Pitchers by Year:")
print("=" * 70)

for year in range(1, YEARS_AHEAD + 1):
    war_col = f'war_year_{year}'
    print(f"\nYear {year} ({BASE_YEAR + year}):")
    
    top_pitchers = pitcher_projections.nlargest(25, war_col)[[
        'playerid', war_col
    ]].copy()
    
    top_pitchers.columns = ['Player ID', f'Year {year} WAR']
    print(top_pitchers.to_string(index=False))
    print()
    print(f"  Top 25 average: {top_pitchers[f'Year {year} WAR'].mean():.2f} WAR")

In [ ]:
# Cell 7: Projection Statistics by Year

print("\nPitcher Projection Statistics:")
print("=" * 70)

for year in range(1, YEARS_AHEAD + 1):
    war_col = f'war_year_{year}'
    
    print(f"\nYear {year} ({BASE_YEAR + year}):")
    print(f"  Total WAR: {pitcher_projections[war_col].sum():.1f}")
    print(f"  Mean WAR: {pitcher_projections[war_col].mean():.2f}")
    print(f"  Median WAR: {pitcher_projections[war_col].median():.2f}")
    print(f"  Std Dev: {pitcher_projections[war_col].std():.2f}")
    print(f"  Min WAR: {pitcher_projections[war_col].min():.2f}")
    print(f"  Max WAR: {pitcher_projections[war_col].max():.2f}")
    
    # Count by tier
    elite = (pitcher_projections[war_col] >= 4.0).sum()
    above_avg = ((pitcher_projections[war_col] >= 2.0) & (pitcher_projections[war_col] < 4.0)).sum()
    average = ((pitcher_projections[war_col] >= 0.0) & (pitcher_projections[war_col] < 2.0)).sum()
    below_avg = (pitcher_projections[war_col] < 0.0).sum()
    
    print(f"\n  Distribution:")
    print(f"    Elite (4+ WAR): {elite}")
    print(f"    Above Average (2-4 WAR): {above_avg}")
    print(f"    Average (0-2 WAR): {average}")
    print(f"    Below Average (<0 WAR): {below_avg}")

## Step 4: Role-Specific Analysis (SP vs RP)

In [ ]:
# Cell 8: Role-Specific Projections

print("\nAverage WAR by Role (Year 1):")
print("=" * 70)

if 'Role' in pitcher_projections.columns:
    role_stats = pitcher_projections.groupby('Role')['war_year_1'].agg([
        ('count', 'count'),
        ('mean', 'mean'),
        ('total', 'sum')
    ]).sort_values('mean', ascending=False)
    
    print(role_stats.to_string())
    
    # Analyze SP vs RP separately
    print("\nStarter vs Reliever Comparison (Year 1):")
    if 'SP' in pitcher_projections['Role'].values:
        sp_avg = pitcher_projections[pitcher_projections['Role'] == 'SP']['war_year_1'].mean()
        print(f"  Starters (SP) average: {sp_avg:.2f} WAR")
    if 'RP' in pitcher_projections['Role'].values:
        rp_avg = pitcher_projections[pitcher_projections['Role'] == 'RP']['war_year_1'].mean()
        print(f"  Relievers (RP) average: {rp_avg:.2f} WAR")
else:
    print("Role information not available in projections.")

## Step 5: Tommy John Surgery Impact Analysis

In [ ]:
# Cell 9: Tommy John Recovery Analysis

print("\nTommy John Surgery Recovery Factors:")
print("=" * 70)
print("\nExpected recovery timeline for pitchers:")
print("  Year 1 post-surgery: 78-90% of pre-injury baseline")
print("  Year 2 post-surgery: 85-97% of baseline")
print("  Year 3+ post-surgery: 93-99% recovery (near full recovery)")
print("\nRecovery varies by:")
print("  - Role (SP vs RP)")
print("  - Age at surgery")
print("  - Pre-injury performance level")
print("\nNote: Tommy John adjustments require injury data.")
print("      Provide injury_records to pipeline for automatic adjustments.")

## Step 6: Save and Export

In [ ]:
# Cell 10: Export Projections

# Projections already saved by pipeline, but can export additional formats

print("\nExporting pitcher projections...")

# Save top 100 to CSV
top_100_path = project_root / f"predictions/top_100_pitchers_{BASE_YEAR + 1}.csv"
top_100 = pitcher_projections.nlargest(100, 'war_year_1')
top_100.to_csv(top_100_path, index=False)
print(f"  Top 100 pitchers saved to: {top_100_path}")

# Save elite pitchers (4+ WAR)
elite_path = project_root / f"predictions/elite_pitchers_{BASE_YEAR + 1}.csv"
elite_pitchers = pitcher_projections[pitcher_projections['war_year_1'] >= 4.0]
elite_pitchers.to_csv(elite_path, index=False)
print(f"  Elite pitchers (4+ WAR) saved to: {elite_path}")
print(f"    Count: {len(elite_pitchers)}")

# Save by role if available
if 'Role' in pitcher_projections.columns:
    for role in ['SP', 'RP']:
        if role in pitcher_projections['Role'].values:
            role_path = project_root / f"predictions/{role.lower()}_projections_{BASE_YEAR + 1}.csv"
            role_projections = pitcher_projections[pitcher_projections['Role'] == role]
            role_projections.to_csv(role_path, index=False)
            print(f"  {role} projections saved to: {role_path}")
            print(f"    Count: {len(role_projections)}")

print("\nExport complete!")

## Summary

Pitcher projections generated successfully!

**Output Files:**
- `predictions/future_projections_pitcher_YYYY.csv` - All pitcher projections
- `predictions/top_100_pitchers_YYYY.csv` - Top 100 projected pitchers
- `predictions/elite_pitchers_YYYY.csv` - Elite (4+ WAR) pitchers
- `predictions/sp_projections_YYYY.csv` - Starting pitcher projections
- `predictions/rp_projections_YYYY.csv` - Relief pitcher projections

**Next Steps:**
- Run hitter projections: See `hitter_future_projections.ipynb`
- Validate projections: See `sWARm_future_deep_dive.ipynb`
- Combine with hitter projections for league-wide zero-sum constraint